# 05 - NLP Interface With IPOPT

This notebook introduces CasADi's nonlinear-programming interface with IPOPT.

Learning goals:

- Build an NLP dictionary with `x`, `f`, optional `g`, and optional `p`.
- Use `x0`, variable bounds, and constraint bounds.
- Read solver outputs and solver statistics.
- Practice writing an NLP formulation before applying it to IK.

In [ ]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "casadi",
    "numpy",
    "matplotlib",

])


In [ ]:
import casadi as ca
import matplotlib.pyplot as plt
import numpy as np

print("CasADi version:", ca.__version__)
ipopt_opts = {
    "ipopt.print_level": 0,
    "ipopt.sb": "yes",
    "print_time": False,
}

## The `nlpsol` Solver Object Contract

A CasADi NLP solver follows the same two-stage pattern as `qpsol`: first build a symbolic solver object, then call it with numeric values.

The construction-time pattern is

```python
nlp = {
    "x": x,  # decision variables, required
    "f": f,  # scalar objective, required
    "g": g,  # constraint expression, optional
    "p": p,  # parameters, optional
}
solver = ca.nlpsol("solver_name", "ipopt", nlp, opts)
```

The general nonlinear optimization template is

$$
\begin{aligned}
\min_x \quad & f(x, p) \\
\text{s.t.} \quad
& x_{\min} \le x \le x_{\max}, \\
& g_{\min} \le g(x, p) \le g_{\max}.
\end{aligned}
$$

### Construction-Time Pieces

| Piece | Meaning |
|---|---|
| `"x"` | Symbolic decision variable vector. |
| `"f"` | Scalar objective expression. It may be nonlinear. |
| `"g"` | Optional vector of equality or inequality constraint expressions. It may be nonlinear. |
| `"p"` | Optional symbolic parameter vector. Parameters are fixed during one solve. |
| solver plugin | Here we use `"ipopt"`. |
| `opts` | Construction-time solver options, for example `{"ipopt.print_level": 0, "ipopt.tol": 1e-8, "print_time": False}`. IPOPT-specific options use the `ipopt.` prefix. |

### Call-Time Inputs

A solver is a CasADi `Function`, so the solve call uses named inputs:

```python
sol = solver(
    x0=x0_value,
    p=p_value,
    lbx=x_lower,
    ubx=x_upper,
    lbg=g_lower,
    ubg=g_upper,
    lam_x0=lam_x_guess,
    lam_g0=lam_g_guess,
)
```

| Input | Meaning |
|---|---|
| `x0` | Initial guess for the nonlinear solver. This is important for NLPs. |
| `p` | Numeric parameter values matching the symbolic `p`. |
| `lbx`, `ubx` | Lower and upper bounds on decision variables. |
| `lbg`, `ubg` | Lower and upper bounds on constraint expressions `g`. Use equal values for equality constraints. |
| `lam_x0`, `lam_g0` | Optional warm starts for variable-bound and constraint multipliers. |

### Outputs

The result dictionary contains:

| Output | Meaning |
|---|---|
| `x` | Optimizer. |
| `f` | Objective value at the optimizer. |
| `g` | Constraint values at the optimizer. |
| `lam_x`, `lam_g`, `lam_p` | Multipliers/sensitivities returned by the solver. |

All examples below use this same interface; only `f`, `g`, bounds, and parameters change.

## Example 1: Unconstrained Rosenbrock Problem

Decision variable:

$$
x = \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}.
$$

Problem:

$$
\begin{aligned}
\min_{x \in \mathbb{R}^2} \quad
& (1 - x_0)^2 + 100(x_1 - x_0^2)^2.
\end{aligned}
$$

In [ ]:
x = ca.SX.sym("x", 2)
rosenbrock = (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

nlp = {"x": x, "f": rosenbrock}
solver = ca.nlpsol("rosenbrock_solver", "ipopt", nlp, ipopt_opts)
sol = solver(x0=[-1.2, 1.0])

print("x* =", sol["x"])
print("f* =", sol["f"])
print("return status:", solver.stats()["return_status"])

## Example 2: Equality Constraint

Decision variable:

$$
w = \begin{bmatrix} w_0 \\ w_1 \\ w_2 \end{bmatrix}.
$$

Problem:

$$
\begin{aligned}
\min_{w \in \mathbb{R}^3} \quad
& \frac{1}{2} w^T w \\
\text{s.t.} \quad
& w_0 + 2w_1 + w_2 - 1 = 0.
\end{aligned}
$$

Use `g` for the equality constraint and set `lbg = ubg = 0`.

In [ ]:
w = ca.SX.sym("w", 3)
f = 0.5 * ca.dot(w, w)
g = ca.vertcat(w[0] + 2 * w[1] + w[2] - 1)

nlp = {"x": w, "f": f, "g": g}
solver_eq = ca.nlpsol("equality_solver", "ipopt", nlp, ipopt_opts)
sol_eq = solver_eq(x0=[0, 0, 0], lbg=[0], ubg=[0])

print("w* =", sol_eq["x"])
print("g(w*) =", sol_eq["g"])

## Example 3: Parameters, Bounds, And Inequalities

Decision variable:

$$
u = \begin{bmatrix} u_0 \\ u_1 \end{bmatrix}.
$$

Parameter:

$$
u_{\mathrm{target}} \in \mathbb{R}^2.
$$

Problem:

$$
\begin{aligned}
\min_{u \in \mathbb{R}^2} \quad
& \frac{1}{2}\|u - u_{\mathrm{target}}\|_2^2 \\
\text{s.t.} \quad
& u_0^2 + u_1^2 \le 1, \\
& -2 \le u_i \le 2, \quad i=0,1.
\end{aligned}
$$

This is a nonlinear inequality-constrained NLP.

In [ ]:
u = ca.SX.sym("u", 2)
target = ca.SX.sym("target", 2)

f = 0.5 * ca.sumsqr(u - target)
g = ca.vertcat(u[0]**2 + u[1]**2)

nlp = {"x": u, "p": target, "f": f, "g": g}
solver_param = ca.nlpsol("parameterized_solver", "ipopt", nlp, ipopt_opts)

sol_param = solver_param(
    x0=[0.0, 0.0],
    p=[2.0, 0.5],
    lbx=[-2.0, -2.0],
    ubx=[2.0, 2.0],
    lbg=[-ca.inf],
    ubg=[1.0],
)

print("u* =", sol_param["x"])
print("circle constraint value =", sol_param["g"])
print("objective =", sol_param["f"])

## Reading The Result Dictionary

The solver returns primal variables, objective value, constraints, and multipliers.

In [ ]:
print("result keys:", sol_param.keys())
print("solver stats keys sample:", list(solver_param.stats().keys())[:8])
print("return status:", solver_param.stats()["return_status"])

## Exercise Formulation

Before moving to inverse kinematics, implement this NLP yourself in a new cell.

Decision variable:

$$
z = \begin{bmatrix} z_0 \\ z_1 \end{bmatrix}.
$$

Parameter:

$$
z_{\mathrm{ref}} = \begin{bmatrix} z_{\mathrm{ref},0} \\ z_{\mathrm{ref},1} \end{bmatrix}.
$$

Problem:

$$
\begin{aligned}
\min_{z \in \mathbb{R}^2} \quad
& \frac{1}{2}\|z - z_{\mathrm{ref}}\|_2^2 + 0.01 z_0^2 \\
\text{s.t.} \quad
& z_0 + z_1 = 0.5, \\
& \sin(z_0) + z_1^2 \le 0.75, \\
& -1 \le z_i \le 2, \quad i=0,1.
\end{aligned}
$$

Acceptance checks:

- Solve from at least two different initial guesses.
- Print the solution, objective, and both constraint values.
- Verify the equality residual is close to zero and the inequality is below `0.75`.

In [ ]:
z_ref_value = [1.0, 0.2]
initial_guesses = ([0.0, 0.0], [1.5, -0.5])
variable_lower_bounds = [-1.0, -1.0]
variable_upper_bounds = [2.0, 2.0]
constraint_lower_bounds = [0.5, -ca.inf]
constraint_upper_bounds = [0.5, 0.75]

print("Starter data prepared. Implement the NLP from the formulation in this cell.")
